In [1]:
# Bias correct PM2.5

In [2]:
import os
import xarray as xr
import warnings
from utils.utils import adjust_longitude
from utils.utils import bilinear_interp

In [ ]:
warnings.filterwarnings('ignore')
# === Path config ===
PM25_DIR = "/glade/work/awells/air_quality/CESM/pm25/annual_pm25/"
# CHANGE OBS_DIR WHEN PM25 OBS ARE AVAILABLE
OBS_DIR = "/glade/work/awells/air_quality/O3_obs/"
SAVE_DIR = "/glade/work/awells/air_quality/CESM/pm25/pm25_bc/"
SCENARIOS = ["ARISE", "SSP245"]

# === Main loop ===
for scenario in SCENARIOS:
    for ens_num in range(1, 11):
        print(f"Processing {scenario}, Ensemble {ens_num:02d}")
        # Load data arrays
        if scenario == "ARISE":
            dates = "2035-2069"
        elif scenario == "SSP245":
            dates = "2020-2069"

        pm25_file = f"PM25_CESM2_{scenario}_{ens_num:02d}_{dates}.nc"
        pm25_path = os.path.join(PM25_DIR, pm25_file)
        # Convert from kg/m3 to µg/m3
        pm25 = xr.open_dataarray(pm25_path)*10**9
        # Remove the unused lev dimension
        pm25 = pm25.drop_vars("lev")

        hist_file = "PM25_CESM2_hist_01_1990-2009.nc"
        hist_path = os.path.join(PM25_DIR, hist_file)
        # Convert from kg/m3 to µg/m3
        hist = xr.open_dataarray(hist_path)*10**9

        # CHANGE TO PM25 OBS WHEN AVAILABLE (USING FOR GRID HERE)
        obs_file = "Delang_BME_OSDMA8_1990_2017.nc"
        obs_path = os.path.join(OBS_DIR, obs_file)
        obs = xr.open_dataset(obs_path)["ozone"]

        # Baseline years for fi_2000 and historical
        base = slice("1990", "2008")
        hist_base = hist.sel(year=base).mean("year")
        obs_base = obs.sel(year=base).mean("year")
        obs_base = obs_base.rename({'longitude': 'lon', 'latitude': 'lat'})

        # Calculate delta
        # delta_fi = adjust_longitude(pm25 / hist_base)

        # Interpolate to the new grid (change pm25 to delta_fi when pm25 obs available)
        regridder = bilinear_interp(pm25, obs_base)
        # change bc_pm25 to ds_delta_fi when pm25 obs available
        bc_pm25 = regridder(pm25)

        # Bias correct delta
        # bc_pm25 = obs_base * ds_delta_fi

        out_file = f"PM25_BC_CESM2_{scenario}_{ens_num:02d}_{dates}.nc"
        out_path = os.path.join(SAVE_DIR, out_file)

        print(f"Saving to {out_path}")
        description = ("Annual mean PM2.5 bias corrected to XXXX "
                       " - scripts by A.F. Wells (2025)")
        bc_pm25.attrs["description"] = description
        bc_pm25.attrs["ensemble_number"] = ens_num
        bc_pm25.attrs["scenario"] = scenario
        bc_pm25.attrs["units"] = "µg/m3"
        bc_pm25.to_netcdf(out_path)

print("All processing complete.")

Processing ARISE, Ensemble 01
Saving to /glade/work/awells/air_quality/CESM/pm25/pm25_bc/PM25_BC_CESM2_ARISE_01_2035-2069.nc
Processing ARISE, Ensemble 02
Saving to /glade/work/awells/air_quality/CESM/pm25/pm25_bc/PM25_BC_CESM2_ARISE_02_2035-2069.nc
Processing ARISE, Ensemble 03
Saving to /glade/work/awells/air_quality/CESM/pm25/pm25_bc/PM25_BC_CESM2_ARISE_03_2035-2069.nc
Processing ARISE, Ensemble 04
Saving to /glade/work/awells/air_quality/CESM/pm25/pm25_bc/PM25_BC_CESM2_ARISE_04_2035-2069.nc
Processing ARISE, Ensemble 05
Saving to /glade/work/awells/air_quality/CESM/pm25/pm25_bc/PM25_BC_CESM2_ARISE_05_2035-2069.nc
Processing ARISE, Ensemble 06
Saving to /glade/work/awells/air_quality/CESM/pm25/pm25_bc/PM25_BC_CESM2_ARISE_06_2035-2069.nc
Processing ARISE, Ensemble 07
Saving to /glade/work/awells/air_quality/CESM/pm25/pm25_bc/PM25_BC_CESM2_ARISE_07_2035-2069.nc
Processing ARISE, Ensemble 08
Saving to /glade/work/awells/air_quality/CESM/pm25/pm25_bc/PM25_BC_CESM2_ARISE_08_2035-2069.nc
